MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

This notebook is local-only: the leading `_` in its filename excludes it from the
Hugging Face export pipeline (`passage export` strips files prefixed with `_`).
It walks through every step of the PASSAGE evaluation protocol introduced in
`paper/neurips/main.tex` (§5.2 Performance Metrics, §5.3 Statistical Reporting,
§5.4 Trustworthiness, App. J Cross-Family Evaluation Protocol) using the
standalone `passage-eval` package. The notebook assumes a CUDA-enabled host
for the throughput cells; the rest of the cells run on CPU.


## What this notebook computes

Mapped 1:1 to the paper's evaluation protocol — every cell cites the section
of `main.tex` it implements:

1. **Primary decoded-path metrics** (six headline metrics, §5.2 / App. J
   Primary Metrics): `cost_regret_median`, `cost_regret_p90`, `path_assd`,
   `path_hausdorff_p95`, `success_rate`, `obstacle_violation_rate`.
2. **Censored penalty** for failed decodes ($K \gg c^{\star}$, §5.2).
3. **Three-seed paired-bootstrap CI** on $\Delta_m = \operatorname{mean}(m_A - m_B)$
   with seeds `{1234, 2026, 3407}` (§5.3 / App. J Statistical Discipline).
4. **Holm--Bonferroni** correction across the six primary metrics (§5.3).
5. **Pareto-dominant / Trade-off / Indistinguishable** decision rule
   (App. J §Decision Rules).
6. **Decoder-readiness diagnostics** (App. B / App. C): segmentation
   `mask_endpoints_in_same_component` and `boundary_f_score`; regression
   `descent_direction_agreement`, `bellman_residual_reachable`,
   `local_ordinal_concordance`, `field_ssi_mae`. These never carry a headline
   claim (Rule 4).
7. **Inference & deployment footprint** (§5.2 / App. J Deployment Footprint):
   forward / decoder / end-to-end percentile latency, throughput,
   peak GPU memory, parameter count, A* expansions ratio, decoder recovery.
8. **Reproducibility sidecar** (App. J Rule 8): a JSON manifest pinning the
   dataset version, seeds, calibration-decisions hash, hardware stack,
   batch size, dtype, and determinism flag for one headline row.

The notebook is a *demo* on synthetic toy paths so it is fully reproducible
without checkpoints. To compute paper-facing numbers, replace the toy inputs
with the predicted/oracle paths emitted by
`passage-vision/paper/notebooks/06_formulation_showdown.ipynb`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Discover the passage-eval source tree (sibling of this notebook's repo).
ROOT = Path('..').resolve()
for candidate in (
    ROOT / 'src' / 'passage-eval' / 'src',
    ROOT / 'passage-eval' / 'src',
    ROOT / 'src',
):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import passage_eval  # noqa: E402
from passage_eval import (
    DEFAULT_SEEDS,
    PRIMARY_METRICS,
    PairwiseDecision,
    apply_censored_penalty,
    astar_expansions_ratio,
    bellman_residual_reachable,
    benchmark_decoder,
    benchmark_end_to_end,
    benchmark_forward,
    boundary_f_score,
    compute_calibration_decisions_hash,
    compute_primary_metrics,
    cost_regret,
    cost_regret_summary,
    decide_pairwise,
    decoder_recovery_rate,
    descent_direction_agreement,
    field_ssi_mae,
    holm_bonferroni,
    local_ordinal_concordance,
    mask_endpoints_in_same_component,
    multiseed_paired_bootstrap,
    obstacle_violation_rate,
    paired_bootstrap_ci,
    paired_bootstrap_delta,
    path_assd,
    path_hausdorff_p95,
    ReproducibilitySidecar,
    success_rate,
)

print(f'passage_eval=={passage_eval.__version__}')
print(f'pre-registered seeds (§5.3): {DEFAULT_SEEDS}')
print('primary metrics (§5.2 / App. J), in declared order:')
for name, family, higher_is_better in PRIMARY_METRICS:
    direction = 'higher_is_better' if higher_is_better else 'lower_is_better'
    print(f'  - {name:<30s} family={family.value:<12s} {direction}')

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    torch = None  # type: ignore[assignment]
    HAS_TORCH = False

HAS_CUDA = bool(HAS_TORCH and torch.cuda.is_available())
if HAS_CUDA:
    DEVICE = torch.device('cuda')
    print(
        f'CUDA: {torch.cuda.get_device_name()} '
        f'({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB)'
    )
elif HAS_TORCH:
    DEVICE = torch.device('cpu')
    print(
        'No CUDA visible — the §5.2 inference cells will run on CPU. '
        'Paper-facing latency rows must be re-run on the declared R1 stack.'
    )
else:
    DEVICE = None
    print(
        'PyTorch is not installed; the §5.2 inference / throughput cells will '
        'be skipped. Install `passage-eval[torch]` to run them.'
    )

## §5.2 — Synthetic toy data

Every metric is computed on the same toy batch so cells can be re-ordered
without breaking dependencies. Replace this cell with a loader that yields
predicted paths, oracle paths, predicted/oracle costs, and obstacle masks for
a real PASSAGE validation split.

In [ ]:
def diagonal_path(n_cells: int, jitter: int = 0, rng: np.random.Generator | None = None) -> np.ndarray:
    rng = rng or np.random.default_rng(0)
    rows = np.arange(n_cells, dtype=np.float64)
    cols = rows.copy()
    if jitter:
        cols = np.clip(cols + rng.integers(-jitter, jitter + 1, size=n_cells), 0, n_cells - 1).astype(np.float64)
    return np.stack([rows, cols], axis=1)


RES = 64
BATCH = 32
rng = np.random.default_rng(1234)

oracle_paths = [diagonal_path(RES, jitter=0) for _ in range(BATCH)]
predicted_paths_method_a = [diagonal_path(RES, jitter=1, rng=rng) for _ in range(BATCH)]
predicted_paths_method_b = [diagonal_path(RES, jitter=2, rng=rng) for _ in range(BATCH)]

oracle_costs = np.full(BATCH, 100.0)
raw_costs_a = oracle_costs * (1.0 + rng.uniform(low=0.0, high=0.04, size=BATCH))
raw_costs_b = oracle_costs * (1.0 + rng.uniform(low=0.02, high=0.10, size=BATCH))

obstacles = [np.zeros((RES, RES), dtype=bool) for _ in range(BATCH)]
obstacles[7][30, 30] = True  # one violation slot

# Method A fails on a couple of samples; method B fails on a few more.
decoder_succeeded_a = np.ones(BATCH, dtype=bool)
decoder_succeeded_a[3] = False  # decode failure
obstacle_violated_a = np.zeros(BATCH, dtype=bool)
obstacle_violated_a[7] = True    # obstacle traversal

decoder_succeeded_b = np.ones(BATCH, dtype=bool)
decoder_succeeded_b[[3, 5]] = False
obstacle_violated_b = np.zeros(BATCH, dtype=bool)
obstacle_violated_b[[7, 11]] = True

starts = [(0, 0)] * BATCH
goals = [(RES - 1, RES - 1)] * BATCH
print(f'Toy batch: {BATCH} samples at {RES}x{RES}')

### §5.2 — Censored penalty for failed decodes

When the canonical decoder fails to connect $s\to g$ or crosses an obstacle,
the predicted cost is replaced by $K \gg c^{\star}$ (default $K = 10^{6} c^{\star}$,
matching the dataset's obstacle penalty). This keeps the regret distribution
well-defined and forwards failure as tail mass rather than as missing data.

In [ ]:
predicted_costs_a = apply_censored_penalty(
    raw_costs_a, oracle_costs,
    decoder_succeeded=decoder_succeeded_a,
    obstacle_violated=obstacle_violated_a,
    k_factor=1e6,
)
predicted_costs_b = apply_censored_penalty(
    raw_costs_b, oracle_costs,
    decoder_succeeded=decoder_succeeded_b,
    obstacle_violated=obstacle_violated_b,
    k_factor=1e6,
)
pd.DataFrame({
    'oracle_cost': oracle_costs,
    'raw_cost_a': raw_costs_a,
    'censored_cost_a': predicted_costs_a,
    'raw_cost_b': raw_costs_b,
    'censored_cost_b': predicted_costs_b,
}).head(10)

## §5.2 — Primary metric bundle

`compute_primary_metrics` returns the six headline metrics plus a few
diagnostic-side fields (book-keeping for App. B/C).

In [ ]:
summary_a = compute_primary_metrics(
    predicted_paths=predicted_paths_method_a,
    oracle_paths=oracle_paths,
    predicted_costs=predicted_costs_a,
    oracle_costs=oracle_costs,
    obstacles=obstacles,
    starts=starts,
    goals=goals,
)
summary_b = compute_primary_metrics(
    predicted_paths=predicted_paths_method_b,
    oracle_paths=oracle_paths,
    predicted_costs=predicted_costs_b,
    oracle_costs=oracle_costs,
    obstacles=obstacles,
    starts=starts,
    goals=goals,
)
pd.DataFrame({'method_a': summary_a, 'method_b': summary_b})

In [ ]:
# Per-sample regret + summary (median / p90 / p95 / mean / geomean).
# After censoring the mean is dominated by the K-penalised samples; the
# median + p90 are robust — the paper's headline.
regret_a = cost_regret(predicted_costs_a, oracle_costs)
regret_b = cost_regret(predicted_costs_b, oracle_costs)
pd.DataFrame({
    'method_a': cost_regret_summary(regret_a).as_dict(),
    'method_b': cost_regret_summary(regret_b).as_dict(),
})

## §5.3 — Statistical discipline (paired bootstrap, three seeds, Holm--Bonferroni)

Before any test-split pass we pre-register seeds `{1234, 2026, 3407}`, the six
primary metrics in declared order, and family-wise $\alpha = 0.05$. Each
method then yields a per-sample primary-metric array per seed; we paired-
bootstrap on $\Delta_m = \operatorname{mean}(m_A - m_B)$, pool 3×$10^4$ draws
across seeds, and apply Holm--Bonferroni across the six primary metrics.

In [ ]:
# Per-seed per-sample arrays. In a paper-facing run these come from the
# three independently-trained checkpoints; here we synthesise three
# noisy replicas of the toy batch so the cell exercises the full path.
def per_sample_metric_arrays(seed: int):
    rng = np.random.default_rng(seed)
    # Per-sample regret values for both methods (already censored).
    rgt_a = cost_regret(predicted_costs_a, oracle_costs)
    rgt_b = cost_regret(predicted_costs_b, oracle_costs)
    # Geometric per-sample arrays.
    assd_a = path_assd(predicted_paths_method_a, oracle_paths)
    assd_b = path_assd(predicted_paths_method_b, oracle_paths)
    haus_a = path_hausdorff_p95(predicted_paths_method_a, oracle_paths)
    haus_b = path_hausdorff_p95(predicted_paths_method_b, oracle_paths)
    # Reliability: success / obstacle-violation as 0/1 per sample.
    succ_a = np.array(decoder_succeeded_a, dtype=float) - rng.uniform(0, 0.01, BATCH)
    succ_b = np.array(decoder_succeeded_b, dtype=float) - rng.uniform(0, 0.01, BATCH)
    viol_a = np.array(obstacle_violated_a, dtype=float)
    viol_b = np.array(obstacle_violated_b, dtype=float)
    return {
        'cost_regret_median':        (rgt_a, rgt_b),
        'cost_regret_p90':           (rgt_a, rgt_b),
        'path_assd':                 (assd_a, assd_b),
        'path_hausdorff_p95':        (haus_a, haus_b),
        'success_rate':              (succ_a, succ_b),
        'obstacle_violation_rate':   (viol_a, viol_b),
    }

by_seed = {s: per_sample_metric_arrays(s) for s in DEFAULT_SEEDS}

deltas: list = []
draws_by_metric: dict[str, np.ndarray] = {}
for name, family, higher_is_better in PRIMARY_METRICS:
    a_by_seed = {s: by_seed[s][name][0] for s in DEFAULT_SEEDS}
    b_by_seed = {s: by_seed[s][name][1] for s in DEFAULT_SEEDS}
    # cost_regret_median / p90 use the median / p90 reduction; everything
    # else uses the paper's $\Delta_m = \mathrm{mean}(m_A - m_B)$.
    if name == 'cost_regret_median':
        statistic = np.median
    elif name == 'cost_regret_p90':
        statistic = lambda x: np.percentile(x, 90.0)
    else:
        statistic = np.mean
    rec, draws = multiseed_paired_bootstrap(
        a_by_seed, b_by_seed,
        metric=name, higher_is_better=higher_is_better,
        statistic=statistic,
        seeds=DEFAULT_SEEDS, n_bootstrap_per_seed=2_000,
        return_draws=True,
    )
    deltas.append(rec)
    draws_by_metric[name] = draws

pd.DataFrame([
    {
        'metric': d.metric,
        'delta': d.delta,
        'ci_low': d.ci_low,
        'ci_high': d.ci_high,
        'excludes_zero': d.excludes_zero,
        'favours_a': d.favours_a,
        'n': d.n,
        'n_bootstrap': d.n_bootstrap,
    } for d in deltas
])

In [ ]:
# Two-sided p-values from the bootstrap draws, then Holm--Bonferroni at
# family-wise alpha = 0.05 across the six primary metrics.
p_values = {d.metric: d.two_sided_p_value(draws_by_metric[d.metric]) for d in deltas}
holm = holm_bonferroni(p_values, alpha=0.05)
pd.DataFrame([
    {
        'metric': h.metric,
        'p_value': h.p_value,
        'p_adjusted': h.p_adjusted,
        'reject_null': h.rejected,
    } for h in holm.values()
])

In [ ]:
# App. J §Decision Rules: reduce per-metric votes to a verdict.
verdict, votes = decide_pairwise(deltas, holm=holm)
print(f'Verdict (method_a vs method_b): {verdict.value}')
pd.DataFrame.from_dict(votes, orient='index', columns=['vote'])

## App. B / App. C — Decoder-readiness diagnostics

These never carry a headline claim (Rule 4 of App. J). They are reported in
App. B (segmentation) / App. C (regression) to attribute primary-metric
regressions to an actual decoder-readiness failure rather than a mask /
field-space accuracy drop.

In [ ]:
# Synthetic segmentation example: a path mask with a topological cut.
mask = np.zeros((RES, RES), dtype=bool)
for r, c in oracle_paths[0].astype(int):
    mask[r, c] = True
mask_cut = mask.copy()
mask_cut[RES // 2, :] = False  # introduce a cut

diag_seg = {
    'mask_endpoints_in_same_component_full': mask_endpoints_in_same_component(mask, (0, 0), (RES - 1, RES - 1)),
    'mask_endpoints_in_same_component_cut': mask_endpoints_in_same_component(mask_cut, (0, 0), (RES - 1, RES - 1)),
    'boundary_f_score@1': boundary_f_score(mask, mask, tolerance=1),
    'boundary_f_score@3_with_cut': boundary_f_score(mask_cut, mask, tolerance=3),
}
pd.Series(diag_seg).to_frame('value')

In [ ]:
# Synthetic regression example: cost-to-go field that decreases towards the goal.
h = w = RES
ii, jj = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
oracle_field = np.sqrt((ii - (h - 1)) ** 2 + (jj - (w - 1)) ** 2).astype(np.float64)
noise = np.random.default_rng(0).standard_normal(oracle_field.shape) * 0.5
predicted_field = oracle_field + noise
cost = np.ones_like(oracle_field)

diag_reg = {
    'descent_direction_agreement': descent_direction_agreement(predicted_field, oracle_field, goal=(h - 1, w - 1)),
    'bellman_residual_reachable': bellman_residual_reachable(predicted_field, cost, goal=(h - 1, w - 1)),
    'local_ordinal_concordance': local_ordinal_concordance(predicted_field, oracle_field),
    'field_ssi_mae': field_ssi_mae(predicted_field, oracle_field),
}
pd.Series(diag_reg).to_frame('value')

## §5.2 — Inference and throughput

The CUDA path is used iff a CUDA device is visible (we checked above). Replace
`ToySegHead` with the network-under-test for paper-facing numbers. Per Rule 7
of App. J, **decoding is part of inference** — `benchmark_end_to_end` is the
row that lands in the headline table; `benchmark_forward` and `benchmark_decoder`
are diagnostic decompositions.

In [ ]:
if HAS_TORCH:
    import torch.nn as nn

    class ToySegHead(nn.Module):
        def __init__(self):
            super().__init__()
            self.body = nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
                nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
                nn.Conv2d(32, 1, 1),
            )

        def forward(self, x):
            return self.body(x)

    model = ToySegHead().to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Toy model params: {n_params/1e6:.4f} M')
else:
    model = None
    print('skipped: torch unavailable')

In [ ]:
if HAS_TORCH:
    rec_fwd_bs1 = benchmark_forward(
        model,
        input_shape=(1, 3, 256, 256),
        n_warmup=5,
        n_timed=20,
        determinism_flag='algorithmically_deterministic',
        dtype='fp16' if HAS_CUDA else 'fp32',
    )
    rec_fwd_bs8 = benchmark_forward(
        model,
        input_shape=(8, 3, 256, 256),
        n_warmup=5,
        n_timed=20,
        determinism_flag='algorithmically_deterministic',
        dtype='fp16' if HAS_CUDA else 'fp32',
    )
    out_df = pd.DataFrame([rec_fwd_bs1.as_dict(), rec_fwd_bs8.as_dict()])
else:
    rec_fwd_bs1 = rec_fwd_bs8 = None
    out_df = pd.DataFrame()
out_df

In [ ]:
from time import sleep


def toy_decoder(logits) -> list[tuple[int, int]]:
    sleep(0.001)
    return [(0, 0), (1, 1)]


if HAS_TORCH:
    logits_batch = [torch.zeros(1, 1, 256, 256, device=DEVICE) for _ in range(64)]
    rec_decoder = benchmark_decoder(toy_decoder, logits_batch, n_warmup=1)
    out = pd.Series(rec_decoder.as_dict()).to_frame('value')
else:
    rec_decoder = None
    out = pd.DataFrame()
out

In [ ]:
if HAS_TORCH:
    dtype = torch.float16 if HAS_CUDA else torch.float32
    fwd_model = model.half() if HAS_CUDA else model
    inputs = [torch.randn(1, 3, 256, 256, device=DEVICE, dtype=dtype) for _ in range(32)]
    rec_e2e = benchmark_end_to_end(
        fwd_model,
        toy_decoder,
        inputs,
        n_warmup=5,
    )
    out = pd.Series(rec_e2e.as_dict()).to_frame('value')
else:
    rec_e2e = None
    out = pd.DataFrame()
out

## §5.4 — Surrogate efficiency (Trustworthiness axis iv)

`xratio = E_pred / E_oracle` and `decoder_recovery_rate` gate whether a
learned surrogate actually accelerates A* end-to-end.

In [ ]:
# Toy expansions counts: surrogate plans on the predicted field, oracle
# is plain A* on the true cost grid. Replace these with real counts
# emitted by the decoder.
rng = np.random.default_rng(0)
expansions_oracle = rng.integers(low=4_000, high=10_000, size=BATCH).astype(np.float64)
expansions_pred = expansions_oracle * rng.uniform(low=0.1, high=0.6, size=BATCH)

ratio = astar_expansions_ratio(expansions_pred, expansions_oracle)
rec = decoder_recovery_rate(decoder_succeeded_a, fallback_triggered=~decoder_succeeded_a & (rng.random(BATCH) < 0.10))
pd.Series({**ratio, 'decoder_recovery_rate': rec}).to_frame('value')

## App. J Rule 8 — Reproducibility sidecar

Every headline row carries a sidecar JSON pinning dataset version, content
hash, seed set, calibration-decisions hash, hardware stack, batch size, dtype,
and determinism flag. This cell shows how to build one for the toy run; for
paper-facing numbers fill in the dataset hash from `croissant.json` and the
calibration decisions from the calibration-split run.

In [ ]:
calibration_decisions = {
    'segmentation_threshold': 0.5,
    'descent_step_size': 1,
    'descent_max_iter': 4 * RES,
    'recovery_fallback': 'astar_on_predicted_field',
    'loss_recipe': 'bce_dice_path_aware',
}
decisions_hash = compute_calibration_decisions_hash(calibration_decisions)

sidecar = ReproducibilitySidecar(
    method_name='toy_dense_segmentation',
    family='dense_segmentation',
    dataset_version='v2.0.0-demo',
    dataset_content_hash='sha256:demo-hash',
    operating_point=f'{RES}x{RES}/elevation',
    seeds=list(DEFAULT_SEEDS),
    split_seed=0,
    calibration_decisions_hash=decisions_hash,
    hardware_stack='R1' if HAS_CUDA else 'cpu_demo',
    batch_size=1,
    dtype='fp16' if HAS_CUDA else 'fp32',
    determinism_flag='algorithmically_deterministic',
    notes='Synthetic toy run — replace before paper submission.',
    extra={'verdict_vs_method_b': verdict.value},
)
print(sidecar.to_json())

## §5.3 — Single-pair paired-bootstrap CI (legacy helper)

`paired_bootstrap_ci` (kept for backward compatibility) computes the 95%
paired-bootstrap CI on the difference of an arbitrary statistic between two
methods. New code should prefer `paired_bootstrap_delta` /
`multiseed_paired_bootstrap` above, which carry the metric name, the
`higher_is_better` flag, and a Holm-ready p-value helper.

In [ ]:
ci = paired_bootstrap_ci(
    lambda a, b: float(np.nanmedian(a) - np.nanmedian(b)),
    regret_a, regret_b,
    n_bootstrap=2_000, seed=0,
)
pd.Series(ci).to_frame('value')

## Export to a paper-facing CSV row

Concatenates the primary bundle, the inference record (forward + decoder + e2e), and the surrogate-efficiency record into a single row matching the schema of
`outputs/paper/results/headline.csv` declared by the App. J reproducibility sidecar.

In [ ]:
row = {
    **summary_a,
    'astar_expansions_ratio_median': ratio['median'],
    'astar_expansions_ratio_p90': ratio['p90'],
    'astar_expansions_ratio_geomean': ratio['geomean'],
    'decoder_recovery_rate': rec,
    'verdict_vs_method_b': verdict.value,
    'sidecar_calibration_hash': decisions_hash,
}
if HAS_TORCH:
    row.update({
        'forward_latency_ms_bs1': rec_fwd_bs1.mean_latency_ms,
        'forward_throughput_sps_bs8': rec_fwd_bs8.throughput_sps,
        'decoder_latency_ms': rec_decoder.mean_latency_ms,
        'end_to_end_p50_ms': rec_e2e.p50_ms,
        'end_to_end_p95_ms': rec_e2e.p95_ms,
        'end_to_end_p99_ms': rec_e2e.p99_ms,
        'peak_gpu_mem_mib': rec_fwd_bs1.peak_gpu_mem_mib,
        'params_M': rec_fwd_bs1.params_M,
        'determinism_flag': rec_fwd_bs1.determinism_flag,
    })
pd.Series(row).to_frame('value')